# 5.0 — Baselines Clasificación Jerárquica (Dos Etapas)

Este notebook implementa un sistema de clasificación jerárquica de dos etapas:

**Etapa 1: Detección binaria de cáncer**
- Objetivo: Maximizar recall de cáncer (≥95%)
- Output: P(cancer) vs P(nonMalignant)

**Etapa 2: Clasificación del tipo de cáncer**
- Solo se aplica a muestras clasificadas como cáncer en etapa 1
- Objetivo: Maximizar F1-macro en 18 tipos de cáncer
- Usa pesos por clase para manejar el desbalanceo

**Ranking de modelos:**
1. Minimizar `test_cancer_fn` (cáncer predicho como nonMalignant)
2. Minimizar `test_cancer_fnr` (tasa de falsos negativos)
3. Maximizar `test_f1_macro`

In [1]:
from __future__ import annotations
from pathlib import Path
import logging
import warnings
from sklearn.exceptions import ConvergenceWarning

import pandas as pd
import numpy as np

from time import perf_counter
from tqdm.auto import tqdm

from genomics_dl.models.train_multiclass import (
    HierarchicalTrainConfig,
    run_hierarchical_training,
    load_parquet,
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)
logging.getLogger("alembic").setLevel(logging.ERROR)
logging.getLogger("alembic.runtime.migration").setLevel(logging.ERROR)
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("sqlalchemy").setLevel(logging.ERROR)
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    message=".*invalid value encountered in divide.*",
)
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    message=".*A worker stopped while some jobs were given to the executor.*",
)

/workspaces/TFM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Rutas y carga de datos

In [2]:
# Paths
DATA_PROCESSED = Path("../data/processed")
TRAIN_PATH = DATA_PROCESSED / "gse183635_tep_tpm_train.parquet"
TEST_PATH = DATA_PROCESSED / "gse183635_tep_tpm_test.parquet"

df_train = pd.read_parquet(TRAIN_PATH)
df_test = pd.read_parquet(TEST_PATH)

df_train.shape, df_test.shape

((1880, 5452), (471, 5452))

### Separación genes vs metadatos

In [3]:
metadata_cols = [
    "Sample ID",
    "Patient_group",
    "Stage",
    "Sex",
    "Age",
    "Sample-supplying institution",
    "Training series",
    "Evaluation series",
    "Validation series",
    "lib.size",
    "classificationScoreCancer",
    "Class_group",
]

# Genes: columnas ENSG...
gene_cols = [c for c in df_train.columns if str(c).startswith("ENSG")]

assert "Class_group" in df_train.columns
assert "Patient_group" in df_train.columns
assert len(gene_cols) > 0
assert set(gene_cols).isdisjoint(set(metadata_cols))

len(gene_cols), gene_cols[:5]

(5440,
 ['ENSG00000000419',
  'ENSG00000000460',
  'ENSG00000000938',
  'ENSG00000001036',
  'ENSG00000001461'])

### Sanity check de etiquetas (train)

In [4]:
df_train["Class_group"].value_counts(dropna=False)

Class_group
Malignant       1302
nonMalignant     578
Name: count, dtype: int64

In [5]:
df_train.loc[df_train["Class_group"].astype(str) == "Malignant", "Patient_group"].value_counts().head(20)

Patient_group
Non-small-cell lung cancer    417
Ovarian cancer                114
Glioma                        113
Pancreatic cancer              93
Breast cancer                  80
Head and neck cancer           79
Cholangiocarcinoma             71
Colorectal cancer              69
Melanoma                       54
Sarcoma                        44
Endometrial cancer             34
Prostate cancer                23
Multiple Myeloma               22
Urothelial cancer              22
Renal cell cancer              20
Hepatocellular carcinoma       19
Lymphoma                       16
Esophageal carcinoma           12
Name: count, dtype: int64

## Experimentos Jerárquicos (sweep)

Grid de configuraciones:
- **Etapa 1**: XGBoost, LightGBM, ExtraTrees
- **Etapa 2**: XGBoost, LightGBM, ExtraTrees
- **Pesos clase etapa 2**: balanced, sqrt, log
- **Recall mínimo etapa 1**: 0.90, 0.95

Ranking:
1. Minimizar `test_cancer_fn` (cáncer predicho como nonMalignant)
2. Minimizar `test_cancer_fnr`
3. Maximizar `test_f1_macro`

In [6]:
def slugify_token(value):
    return str(value).replace(".", "p").replace("-", "m")

def fmt_secs(s: float) -> str:
    s = int(max(0, s))
    h = s // 3600
    m = (s % 3600) // 60
    ss = s % 60
    if h > 0:
        return f"{h:d}h {m:02d}m {ss:02d}s"
    if m > 0:
        return f"{m:d}m {ss:02d}s"
    return f"{ss:d}s"

In [ ]:
# Configuración base (optimizada para sweep rápido)
BASE_CONFIG = dict(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    class_group_col="Class_group",
    patient_group_col="Patient_group",
    nonmalignant_label="nonMalignant",
    malignant_label="Malignant",
    use_pca=False,
    var_quantile=0.15,
    selector_on_log=False,
    variance_filter_threshold=1e-6,
    cv_splits=5,  # Reducido para sweep (antes: 8)
    random_state=42,
    experiment_name="gse183635_hierarchical",
    save_local_bundle=False,  # Solo guardar el mejor al final
    save_plots=False,
    mlflow_log_artifacts=False,
    mlflow_log_model=False,
)

# Clasificadores etapa 1 (detección binaria)
STAGE1_CLASSIFIERS = [
    ("xgboost", dict(n_estimators=400, max_depth=6, learning_rate=0.1)),
    ("lightgbm", dict(n_estimators=400, max_depth=6, learning_rate=0.1)),
]

# Clasificadores etapa 2 (tipo de cáncer)
STAGE2_CLASSIFIERS = [
    ("xgboost", dict(n_estimators=400, max_depth=8, learning_rate=0.05)),
    ("lightgbm", dict(n_estimators=400, max_depth=8, learning_rate=0.05)),
]

# Diccionarios para acceso rápido (con n_estimators completos para modelo final)
STAGE1_CLASSIFIERS_DICT = {
    "xgboost": dict(n_estimators=650, max_depth=6, learning_rate=0.1),
    "lightgbm": dict(n_estimators=650, max_depth=6, learning_rate=0.1),
}
STAGE2_CLASSIFIERS_DICT = {
    "xgboost": dict(n_estimators=800, max_depth=8, learning_rate=0.05),
    "lightgbm": dict(n_estimators=800, max_depth=8, learning_rate=0.05),
}

# Estrategias de pesos por clase para etapa 2
STAGE2_WEIGHTINGS = ["balanced", "sqrt", "log"]

# Recall mínimo para etapa 1
STAGE1_MIN_RECALLS = [0.90]

# Construir sweep
sweep = []
for s1_name, s1_params in STAGE1_CLASSIFIERS:
    for s2_name, s2_params in STAGE2_CLASSIFIERS:
        for s2_weighting in STAGE2_WEIGHTINGS:
            for s1_min_recall in STAGE1_MIN_RECALLS:
                sweep.append(dict(
                    s1_name=s1_name,
                    s1_params=s1_params,
                    s2_name=s2_name,
                    s2_params=s2_params,
                    s2_weighting=s2_weighting,
                    s1_min_recall=s1_min_recall,
                ))

print(f"Total combinaciones: {len(sweep)}")

Total combinaciones: 12


In [8]:
results = []
errors = []

total = len(sweep)
start_all = perf_counter()

# Calculadora de tiempo por iteración con media móvil
ema = None
alpha = 0.25
done = 0

pbar = tqdm(sweep, total=total, desc="Sweep Jerárquico", unit="run")

for combo in pbar:
    t0 = perf_counter()

    s1_name = combo["s1_name"]
    s1_params = combo["s1_params"]
    s2_name = combo["s2_name"]
    s2_params = combo["s2_params"]
    s2_weighting = combo["s2_weighting"]
    s1_min_recall = combo["s1_min_recall"]

    model_name = f"hier_{s1_name}_{s2_name}"
    model_version = f"w{s2_weighting}_r{int(s1_min_recall*100)}"

    cfg = HierarchicalTrainConfig(
        **BASE_CONFIG,
        model_name=model_name,
        model_version=model_version,
        stage1_clf_name=s1_name,
        stage1_clf_params=s1_params,
        stage1_min_recall=s1_min_recall,
        stage2_clf_name=s2_name,
        stage2_clf_params=s2_params,
        stage2_class_weighting=s2_weighting,
    )

    try:
        res = run_hierarchical_training(cfg, feature_cols=gene_cols)
        tm = res["test_metrics"]
        results.append({
            "model_name": model_name,
            "model_version": model_version,
            "stage1_clf": s1_name,
            "stage2_clf": s2_name,
            "stage2_weighting": s2_weighting,
            "stage1_min_recall": s1_min_recall,
            "test_cancer_fn": tm["cancer_fn"],
            "test_cancer_fnr": tm["cancer_fnr"],
            "test_cancer_recall": tm["cancer_recall_sensitivity"],
            "test_cancer_precision": tm["cancer_precision"],
            "test_f1_macro": tm["f1_macro"],
            "test_f1_weighted": tm["f1_weighted"],
            "test_accuracy": tm["accuracy"],
            "test_balanced_accuracy": tm["balanced_accuracy"],
            "threshold": res["chosen_cancer_threshold"],
            "mlflow_run_id": res["mlflow_run_id"],
            "error": None,
        })
    except Exception as e:
        errors.append({
            "model_name": model_name,
            "model_version": model_version,
            "stage1_clf": s1_name,
            "stage2_clf": s2_name,
            "stage2_weighting": s2_weighting,
            "stage1_min_recall": s1_min_recall,
            "error": repr(e),
        })

    dt = perf_counter() - t0
    ema = dt if ema is None else (alpha * dt + (1 - alpha) * ema)

    done += 1
    elapsed = perf_counter() - start_all
    remaining = (total - done) * (ema if ema is not None else 0.0)

    pbar.set_postfix({
        "last": fmt_secs(dt),
        "avg": fmt_secs(ema),
        "elapsed": fmt_secs(elapsed),
        "eta": fmt_secs(remaining),
        "ok": len(results),
        "err": len(errors),
    })

# DataFrames finales
res_df = (
    pd.DataFrame(results)
      .sort_values(["test_cancer_fn", "test_cancer_fnr", "test_f1_macro"],
                   ascending=[True, True, False])
      .reset_index(drop=True)
)

err_df = pd.DataFrame(errors).reset_index(drop=True)

print(f"OK: {len(res_df)}, Errores: {len(err_df)}")

Sweep Jerárquico:  50%|█████     | 6/12 [30:58<29:26, 294.46s/run, last=4m 39s, avg=5m 03s, elapsed=30m 57s, eta=30m 18s, ok=6, err=0]    /workspaces/TFM/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Sweep Jerárquico: 100%|██████████| 12/12 [2:44:04<00:00, 820.38s/run, last=21m 25s, avg=18m 56s, elapsed=2h 44m 03s, eta=0s, ok=12, err=0]      

OK: 12, Errores: 0


### Resultados del sweep

In [9]:
if len(err_df) > 0:
    print("Errores encontrados:")
    display(err_df)

## Entrenamiento final del mejor modelo

Seleccionamos el mejor modelo del sweep y lo entrenamos con guardado completo:
- Bundle local en `models/`
- Plots en `reports/figures/hierarchical/`
- Artefactos en MLflow

In [10]:
# Elegimos el mejor del sweep
best = res_df.iloc[0].to_dict()
print("Mejor configuración:")
for k, v in best.items():
    if k not in ["mlflow_run_id", "error"]:
        print(f"  {k}: {v}")

Mejor configuración:
  model_name: hier_lightgbm_lightgbm
  model_version: wsqrt_r90
  stage1_clf: lightgbm
  stage2_clf: lightgbm
  stage2_weighting: sqrt
  stage1_min_recall: 0.9
  test_cancer_fn: 33
  test_cancer_fnr: 0.10122699386503067
  test_cancer_recall: 0.8987730061349694
  test_cancer_precision: 0.8517441860465116
  test_f1_macro: 0.35601896274585376
  test_f1_weighted: 0.5237433186212658
  test_accuracy: 0.5414012738853503
  test_balanced_accuracy: 0.34302591148039613
  threshold: 0.5249443025698323


In [11]:
# Configuración del mejor modelo con guardado completo
BEST_CONFIG = HierarchicalTrainConfig(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    class_group_col="Class_group",
    patient_group_col="Patient_group",
    nonmalignant_label="nonMalignant",
    malignant_label="Malignant",
    use_pca=False,
    var_quantile=0.15,
    selector_on_log=False,
    variance_filter_threshold=1e-6,
    cv_splits=10,  # CV completo para modelo final
    random_state=42,
    experiment_name="gse183635_hierarchical",
    model_name="hierarchical_final",
    model_version="v0.1.0",
    stage1_clf_name=best["stage1_clf"],
    stage1_clf_params=STAGE1_CLASSIFIERS_DICT[best["stage1_clf"]],
    stage1_min_recall=best["stage1_min_recall"],
    stage2_clf_name=best["stage2_clf"],
    stage2_clf_params=STAGE2_CLASSIFIERS_DICT[best["stage2_clf"]],
    stage2_class_weighting=best["stage2_weighting"],
    # Activar guardado
    save_local_bundle=True,
    save_plots=True,
    mlflow_log_artifacts=True,
    mlflow_log_model=True,
    output_models_dir="models",
    output_figures_dir="reports/figures/hierarchical",
)

print("Entrenando modelo final con CV completo...")
final_result = run_hierarchical_training(BEST_CONFIG, feature_cols=gene_cols)
print(f"\nModelo guardado en: {final_result['bundle_dir']}")

Entrenando modelo final con CV completo...

Modelo guardado en: /workspaces/TFM/models/hierarchical_final/v0.1.0


In [12]:
# Mostrar métricas finales
print("MÉTRICAS MODELO FINAL")

print("\n--- Métricas de Test ---")
tm = final_result["test_metrics"]
print(f"  Accuracy:           {tm['accuracy']:.4f}")
print(f"  Balanced Accuracy:  {tm['balanced_accuracy']:.4f}")
print(f"  F1-macro:           {tm['f1_macro']:.4f}")
print(f"  F1-weighted:        {tm['f1_weighted']:.4f}")

print("\n--- Detección de Cáncer (Etapa 1) ---")
print(f"  Umbral:             {final_result['chosen_cancer_threshold']:.4f}")
print(f"  Recall (Sens.):     {tm['cancer_recall_sensitivity']:.4f}")
print(f"  Especificidad:      {tm['cancer_specificity']:.4f}")
print(f"  Precisión:          {tm['cancer_precision']:.4f}")
print(f"  FNR:                {tm['cancer_fnr']:.4f}")
print(f"  TP: {tm['cancer_tp']}, FP: {tm['cancer_fp']}, TN: {tm['cancer_tn']}, FN: {tm['cancer_fn']}")

print("\n--- ROC/PR AUC ---")
print(f"  ROC AUC:            {tm['cancer_roc_auc']:.4f}")
print(f"  PR AUC:             {tm['cancer_pr_auc']:.4f}")

print("\n" + "="*60)

MÉTRICAS MODELO FINAL

--- Métricas de Test ---
  Accuracy:           0.5520
  Balanced Accuracy:  0.3483
  F1-macro:           0.3636
  F1-weighted:        0.5358

--- Detección de Cáncer (Etapa 1) ---
  Umbral:             0.5766
  Recall (Sens.):     0.8988
  Especificidad:      0.6690
  Precisión:          0.8592
  FNR:                0.1012
  TP: 293, FP: 48, TN: 97, FN: 33

--- ROC/PR AUC ---
  ROC AUC:            0.8756
  PR AUC:             0.9350

